In [ ]:
# Determines whether stoichiometric model's biomass reaction contains metabolites used in PTM reactions (e.g., cofactors), to avoid overestimating demand for those metabolites.
# before running the script, make sure SM_biomass_id is assigned correctly.
import cobra,json,os,sys,re,pandas as pd, io, requests, ast
from bs4 import BeautifulSoup
from collections import OrderedDict
from copy import deepcopy

# keep going up until the "pycore" folder is found
wd = os.getcwd()
while not os.path.exists('pycore'):
	os.chdir('..')
base_path = os.getcwd()
sys.path.append(base_path)
from pycore import *

os.chdir(wd)

/Users/ejm6426/Documents/pkRBA/pycore/../build_model/input/iIsor850_RBA.json


In [2]:
# read protein_stoich file
df_pro = read_spreadsheet('./input/PROTEIN_stoich_curation.tsv')

In [3]:
# find all metabolites in PTM_stoich_edits column; split by | and then by :, taking the first part as the metabolite ID
ptm_mets = set()
ptm_chebi_ids = set()
# accounts for dicts in PTM_stoich_edits column, which are represented as strings; convert to dict and combine the values into a single string
df_pro['PTM_stoich_edits'] = df_pro['PTM_stoich_edits'].apply(lambda x: '|'.join(ast.literal_eval(x).values()) if "{" in str(x) and "}" in str(x) else x)
for entry in df_pro['PTM_stoich_edits']:
	if pd.isna(entry):
		continue
	# check if entry can be represented as a dict; if so, convert to dict and combine the values into the edits list
	edits = '|'.join(entry.values()) if type(entry) == dict else entry.split('|')
	for edit in edits:
		met_id = edit.split(':')[0]
		ptm_mets.add(met_id)
		if met_id not in model.metabolites:
			continue
		met_chebi_id = model.metabolites.get_by_id(met_id).annotation.get('chebi',None)
		if met_chebi_id:
			ptm_chebi_ids.add(met_chebi_id)
		
# find biomass reaction, and all compounds involved in it
PTM_mets_to_review = set()
PTM_mets_relations = dict()
biomass_rxn = model.reactions.get_by_id(RBA_defaults.SM_biomass_id)
for met in biomass_rxn.metabolites:
	if met.id in ptm_mets or met.annotation.get('chebi',None) in ptm_chebi_ids:
		print(f"Metabolite {met.id} is a biomass component used in PTMs.")
		# add all metabolites with the same ChEBI ID to PTM_mets_to_review
		if met.annotation.get('chebi',None) in ptm_chebi_ids:
			chebi_id = met.annotation.get('chebi',None)
			for m in model.metabolites:
				if m.annotation.get('chebi',None) == chebi_id:
					PTM_mets_to_review.add(m.id)
					PTM_mets_relations.setdefault(met.id,set()).add(m.id)
		else:
			PTM_mets_to_review.add(met.id)
			PTM_mets_relations.setdefault(met.id,set()).add(met.id)

# check if model can grow
model.objective = biomass_rxn
solution = model.optimize()
if solution.status != 'optimal' or solution.objective_value < 1e-6:
	raise Warning("Warning: model cannot grow optimally. Please check biomass reaction and PTM metabolites added.")
print(f"Model can grow optimally before removing genes requiring PTM metabolites to review. Growth rate: {solution.objective_value}")

df_pro['PTM_stoich_edits'] = df_pro['PTM_stoich_edits'].apply(lambda x: x.split('|') if type(x) == str and '|' in x else x)
df_pro = df_pro.explode('PTM_stoich_edits')
# convert all entries in PTM_stoich_edits to dicts, splitting by ',' for key-value pairs, and ':' for key and value
df_pro['PTM_stoich_edits'] = df_pro['PTM_stoich_edits'].apply(lambda x: {item.split(':')[0]:int(item.split(':')[1]) for item in x.split(',')} if type(x) == str else x)

Metabolite ca2_c is a biomass component used in PTMs.
Metabolite cu2_c is a biomass component used in PTMs.
Metabolite fe2_c is a biomass component used in PTMs.
Metabolite h2o_c is a biomass component used in PTMs.
Metabolite h_c is a biomass component used in PTMs.
Metabolite k_c is a biomass component used in PTMs.
Metabolite mg2_c is a biomass component used in PTMs.
Metabolite mn2_c is a biomass component used in PTMs.
Metabolite zn2_c is a biomass component used in PTMs.
Model can grow optimally before removing genes requiring PTM metabolites to review. Growth rate: 0.781531866211586


In [4]:
# make separate df for proteins that don't require PTM mets to review
essential_mets = set()
essential_met_types = set() # e.g., metabolites that are essential only when all forms are knocked out
for met in PTM_mets_to_review:
	df_pro_no_ptm_review = df_pro[~df_pro['PTM_stoich_edits'].apply(lambda x: any(met2 == met for met2 in x.keys()) if type(x) == dict else False)]
	# test if met is essential by knocking out all genes that use it in PTMs and checking growth
	with model as m:
		# disable all genes which require PTM_mets_to_review
		for gene in m.genes:
			if gene.id not in df_pro_no_ptm_review.gene_src.values and gene.id in df_pro.gene_src.values:
				gene.knock_out()
		solution = m.optimize()
		if solution.status != 'optimal' or solution.objective_value < 1e-6:
			# print(f"Warning: model cannot grow optimally after removing genes requiring {met}. Please check biomass reaction and PTM metabolites added.")
			essential_mets.add(met)
		else:
			# test knocking out genes using other versions of the same metabolite simultaneously
			related_mets = PTM_mets_relations.get(met,set())
			for related_met in related_mets:
				if related_met == met:
					continue
				df_pro_no_ptm_review_related = df_pro[~df_pro['PTM_stoich_edits'].apply(lambda x: any(related_met2 == related_met for related_met2 in x.keys()) if type(x) == dict else False)]
				for gene in m.genes:
					if gene.id not in df_pro_no_ptm_review_related.gene_src.values and gene.id in df_pro.gene_src.values:
						gene.knock_out()
			solution = m.optimize()
			if solution.status != 'optimal' or solution.objective_value < 1e-6:
				essential_met_types.add(frozenset(PTM_mets_relations.get(met,set())))

			# print(f"Model can grow optimally after removing genes requiring {met}. Growth rate: {solution.objective_value}")

print(f"PTM metabolites that are essential individually: {essential_mets}")
print(f"Essential PTM metabolite types (groups), including forms in all compartments: {essential_met_types}")

/Applications/anaconda3/envs/rba/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


PTM metabolites that are essential individually: {'k_c', 'zn2_c', 'fe2_c', 'mg2_c', 'mn2_c'}
Essential PTM metabolite types (groups), including forms in all compartments: set()


In [ ]:
# create biomass composition file as draft for user

# Amino acid info
from unittest import case

df_aamap = pd.read_csv('./input/PROTEIN_amino_acid_map.txt', sep='\t')
df_aamap.index = df_aamap.aa_abbv.to_list()
# remove "MET-" prefix from tRNA_in and tRNA_out columns
df_aamap['tRNA_in'] = df_aamap['tRNA_in'].apply(lambda x: x.replace('MET-','') if type(x) == str else x)
df_aamap['tRNA_out'] = df_aamap['tRNA_out'].apply(lambda x: x.replace('MET-','') if type(x) == str else x)

met_relations = dict()
# find versions of tRNA_in and tRNA_out for each amino acid (in all compartments)
for aa in df_aamap.aa_abbv:
	tRNA_in = df_aamap.loc[aa, 'tRNA_in']
	tRNA_out = df_aamap.loc[aa, 'tRNA_out']
	met_relations[tRNA_in] = tRNA_in
	met_relations[tRNA_out] = tRNA_out
	for met in model.metabolites:
		if met.annotation.get('chebi',None) == model.metabolites.get_by_id(tRNA_in).annotation.get('chebi',None) != None:
			met_relations[met.id] = tRNA_in
		elif met.annotation.get('chebi',None) == model.metabolites.get_by_id(tRNA_out).annotation.get('chebi',None) != None:
			met_relations[met.id] = tRNA_out
# use "MW" column to determine fraction of biomass comprised of proteins (assuming biomass MW is 1 g/mmol), counting only tRNA_in; and create a JSON file with the biomass composition altered to replace protein metabolites (inc. tRNA_in and tRNA_out) with "BIO-prot"
new_biomass_composition = dict()
gam_val = biomass_rxn.metabolites.get(model.metabolites.get_by_id('adp_c'),0)
non_gam_coeff = get_coeff_without_gam(model, biomass_rxn.id, gam_val)
biomass_mw = 0
biomass_AA_fractions = {}
for met, coeff in biomass_rxn.metabolites.items():
	if met.id in met_relations and met_relations[met.id] in df_aamap.tRNA_out.values:
		continue
	elif met.id in met_relations and met_relations[met.id] in df_aamap.tRNA_in.values:
		new_biomass_composition['BIO-prot'] = new_biomass_composition.get('BIO-prot', 0) + (coeff * df_aamap.loc[df_aamap.tRNA_in == met_relations[met.id], 'MW'].values[0] / 1000)
		biomass_AA_fractions[df_aamap.loc[df_aamap.tRNA_in == met_relations[met.id], 'aa_abbv'].values[0]] = coeff
	else:
		# new_biomass_composition[f"MET-{met.id}"] = non_gam_coeff.get(met.id, 0)
		new_biomass_composition[f"MET-{met.id}"] = coeff
		if coeff < 0:
			if met.formula_weight is not None:
				biomass_mw -= non_gam_coeff[met.id] * float(met.formula_weight) / 1000
			else:
				print(f"Warning: no formula weight available for metabolite {met.id}; excluding it from biomass MW.")
biomass_mw -= new_biomass_composition.get('BIO-prot', 0)
# remove any metabolites with zero coefficients from new_biomass_composition
new_biomass_composition = {met: coeff for met, coeff in new_biomass_composition.items() if abs(coeff) != 0}

with open('./input/v_trans_AA.txt', 'w') as f:
	f.write('/\n')
	for aa, frac in sorted(biomass_AA_fractions.items()):
		f.write(f"'{aa}' {frac/sum(biomass_AA_fractions.values())}\n")
	f.write('/\n')
# write to YAML file, adding text at the beginning to explain what this file is and how to use it
with open('./input/biomass_composition.yaml', 'w') as f:
	f.write("# This file contains the biomass composition of the model (each unindented key is a rxn, each indented key is a metabolite), with the following changes:\n#  'MET-' prefixed metabolites and protein metabolites replaced by 'BIO-prot' (with a coefficient based on the molecular weight of the amino acids) to allow flexibility in proteins used.\n")
	f.write(f"# Note: Review this file carefully (and the rxn {biomass_rxn.id}, for comparison) before using it in your model. If other aspects of biomass composition should vary (e.g., carbohydrate fraction varying based on growth rate), see the documentation for more information.\n\n")
	f.write(f"# For reference, here are automatically predicted values for:\n")
	f.write(f"#   GAM (growth-associated maintenance): {gam_val} mmol ATP/gDW\n#   biomass MW: {biomass_mw} g/mmol (should be close to 1 g/mmol; if not, confirm that GAM was accounted for properly and that all biomass components have formula weights available).\n")
	f.write("BIOSYN-PROTTOBIO:\n  BIO-protmodeled: -1\n  BIO-prot: 1\n")
	f.write(f"{RBA_defaults.RBA_biomass_id}:")
	for met, coeff in new_biomass_composition.items():
		f.write(f"\n  {met}: {coeff}")